# lasso_SPD vs lasso_SPD_numba Equivalence Tests

This notebook compares functions in `DiffusionRWR_model_package.graph_generation.lasso_SPD` and `DiffusionRWR_model_package.graph_generation.lasso_SPD_numba` to verify they produce equivalent outputs within floating-point tolerance.

In [2]:
from pathlib import Path
import sys
import traceback

import numpy as np
import pandas as pd

# Resolve repository root for robust imports.
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
project_root = None
for c in candidates:
    if (c / 'DiffusionRWR_model_package').exists():
        project_root = c
        break

if project_root is None:
    raise RuntimeError('Could not locate project root containing DiffusionRWR_model_package')

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from DiffusionRWR_model_package.graph_generation import lasso_SPD as base
from DiffusionRWR_model_package.graph_generation import lasso_SPD_numba as numba_mod

print(f'Project root: {project_root}')
print('Loaded modules: base=lasso_SPD, numba=lasso_SPD_numba')

Number of CSV files found: 4

CSV files found:
  - overlap_filtered_k20me3_m_v2.csv
  - overlap_filtered_k27me3_m_v2.csv
  - overlap_filtered_k9me2_m_v2.csv
  - overlap_filtered_rna_ai_m_v2.csv
  Filtered to integer time points: ['0', '1', '2', '3', '4']
Imported 'overlap_filtered_k20me3_m_v2' with shape: (657, 5) (row-wise standardized)
  Filtered to integer time points: ['0', '1', '2', '3', '4']
Imported 'overlap_filtered_k27me3_m_v2' with shape: (650, 5) (row-wise standardized)
  Filtered to integer time points: ['0', '1', '2', '3', '4']
Imported 'overlap_filtered_k9me2_m_v2' with shape: (1001, 5) (row-wise standardized)
  Filtered to integer time points: ['0', '1', '2', '3', '4']
Imported 'overlap_filtered_rna_ai_m_v2' with shape: (384, 5) (row-wise standardized)

Successfully imported 4 datasets.
Available datasets: ['overlap_filtered_k20me3_m_v2', 'overlap_filtered_k27me3_m_v2', 'overlap_filtered_k9me2_m_v2', 'overlap_filtered_rna_ai_m_v2']
Project root: c:\Users\inigo\OneDrive\D

In [3]:
# Utility helpers for equivalence checks.
def max_abs_diff(a, b):
    aa = np.asarray(a, dtype=float)
    bb = np.asarray(b, dtype=float)
    if aa.shape != bb.shape:
        return np.inf
    return float(np.max(np.abs(aa - bb))) if aa.size else 0.0

def compare_arrays(a, b, atol=1e-10, rtol=1e-8):
    aa = np.asarray(a, dtype=float)
    bb = np.asarray(b, dtype=float)
    same_shape = aa.shape == bb.shape
    allclose = same_shape and bool(np.allclose(aa, bb, atol=atol, rtol=rtol, equal_nan=True))
    mad = max_abs_diff(aa, bb) if same_shape else np.inf
    return allclose, mad, aa.shape, bb.shape

def compare_dict_numeric(d1, d2, atol=1e-10, rtol=1e-8):
    keys_ok = set(d1.keys()) == set(d2.keys())
    if not keys_ok:
        return False, {'missing_or_extra_keys': (set(d1.keys()), set(d2.keys()))}

    details = {}
    ok = True
    for k in d1:
        v1, v2 = d1[k], d2[k]
        if isinstance(v1, (int, float, np.number)) and isinstance(v2, (int, float, np.number)):
            close = bool(np.isclose(v1, v2, atol=atol, rtol=rtol, equal_nan=True))
            if not close:
                ok = False
            details[k] = {'base': float(v1), 'numba': float(v2), 'close': close}
        else:
            equal = v1 == v2
            if not equal:
                ok = False
            details[k] = {'base': v1, 'numba': v2, 'equal': equal}

    return ok, details

results = []

def log_result(name, passed, metric='', notes=''):
    results.append({
        'function': name,
        'passed': bool(passed),
        'metric': metric,
        'notes': notes,
    })

In [4]:
# Shared deterministic inputs.
rng = np.random.default_rng(20260405)
X = rng.normal(size=(36, 12))
omega = rng.normal(size=(12, 12))
omega = (omega + omega.T) / 2
np.fill_diagonal(omega, np.abs(np.diag(omega)) + 1.0)
tau = rng.uniform(0.4, 1.3, size=12)
tau_sq = rng.uniform(0.4, 1.5, size=12)
U = rng.normal(size=(12, 12))
U = (U + U.T) / 2
grad = rng.normal(size=(12, 12))
grad = (grad + grad.T) / 2
Z = rng.normal(size=(12, 12))
lam = 0.07
gamma = 0.02
eta = 0.6

data_folder = project_root / 'DiffusionRWR_model_package' / 'data' / 'Modelled'
print('Prepared synthetic and dataset-backed inputs.')

Prepared synthetic and dataset-backed inputs.


In [5]:
# Compare core function outputs one by one.
try:
    r1 = base.test_adjacency_symmetry(omega)
    r2 = numba_mod.test_adjacency_symmetry(omega)
    ok, _ = compare_dict_numeric(r1, r2)
    log_result('test_adjacency_symmetry', ok, notes=str(r1))
except Exception:
    log_result('test_adjacency_symmetry', False, notes=traceback.format_exc())

try:
    r1 = base.test_adjacency_positive_semidefinite(omega)
    r2 = numba_mod.test_adjacency_positive_semidefinite(omega)
    ok, _ = compare_dict_numeric(r1, r2)
    log_result('test_adjacency_positive_semidefinite', ok, notes=f"min_eig_base={r1['min_eigenvalue']:.3e}")
except Exception:
    log_result('test_adjacency_positive_semidefinite', False, notes=traceback.format_exc())

try:
    r1 = base.get_adjacency_element_range(omega)
    r2 = numba_mod.get_adjacency_element_range(omega)
    ok, _ = compare_dict_numeric(r1, r2)
    log_result('get_adjacency_element_range', ok)
except Exception:
    log_result('get_adjacency_element_range', False, notes=traceback.format_exc())

for fn_name, args in [
    ('f', (omega, X, tau)),
    ('g', (omega, tau, lam)),
    ('PI', (omega,)),
    ('gradient_f', (omega, X, tau_sq)),
    ('next_omega', (omega, gamma, U, grad)),
    ('prox_g_over_eta', (Z, tau_sq, lam, eta)),
    ('dual_update', (U, omega, omega + 0.01 * np.eye(12), grad, grad * 1.01, eta, gamma, tau_sq, lam)),
]:
    try:
        out1 = getattr(base, fn_name)(*args)
        out2 = getattr(numba_mod, fn_name)(*args)

        if np.isscalar(out1) and np.isscalar(out2):
            ok = bool(np.isclose(out1, out2, atol=1e-10, rtol=1e-8, equal_nan=True))
            metric = f"abs_diff={abs(float(out1) - float(out2)):.3e}"
        else:
            ok, mad, s1, s2 = compare_arrays(out1, out2, atol=1e-10, rtol=1e-8)
            metric = f"max_abs_diff={mad:.3e}, shape_base={s1}, shape_numba={s2}"

        log_result(fn_name, ok, metric=metric)
    except Exception:
        log_result(fn_name, False, notes=traceback.format_exc())

try:
    o1, t1 = base.initialize_omega_lasso(X, lam=0.08, max_iter_inner=2000)
    o2, t2 = numba_mod.initialize_omega_lasso(X, lam=0.08, max_iter_inner=2000)
    ok_o, mad_o, _, _ = compare_arrays(o1, o2, atol=1e-8, rtol=1e-6)
    ok_t, mad_t, _, _ = compare_arrays(t1, t2, atol=1e-8, rtol=1e-6)
    log_result('initialize_omega_lasso', ok_o and ok_t, metric=f"omega_diff={mad_o:.3e}, tau_diff={mad_t:.3e}")
except Exception:
    log_result('initialize_omega_lasso', False, notes=traceback.format_exc())

In [6]:
# Compare proximal solver outputs and dataset-backed wrappers.
try:
    omega1, losses1 = base.proximal_precision(
        X, lam=0.05, gamma=0.01, eta=0.5, max_iter=10, return_omega=True, show_ls_loss=False
    )
    omega2, losses2 = numba_mod.proximal_precision(
        X, lam=0.05, gamma=0.01, eta=0.5, max_iter=10, return_omega=True, show_ls_loss=False
    )
    ok_o, mad_o, _, _ = compare_arrays(omega1, omega2, atol=1e-8, rtol=1e-6)
    ok_l, mad_l, _, _ = compare_arrays(losses1, losses2, atol=1e-8, rtol=1e-6)
    log_result('proximal_precision(return_omega=True)', ok_o and ok_l, metric=f"omega_diff={mad_o:.3e}, loss_diff={mad_l:.3e}")
except Exception:
    log_result('proximal_precision(return_omega=True)', False, notes=traceback.format_exc())

try:
    Q1, losses1, meta1 = base.run_proximal_on_rna_data(
        folder_path=data_folder, rna_key_contains='rna', top_n_genes=40, lam=0.08, gamma=0.01, eta=0.5, max_iter=8, make_plot=False
    )
    Q2, losses2, meta2 = numba_mod.run_proximal_on_rna_data(
        folder_path=data_folder, rna_key_contains='rna', top_n_genes=40, lam=0.08, gamma=0.01, eta=0.5, max_iter=8, make_plot=False
    )

    ok_q, mad_q, _, _ = compare_arrays(Q1, Q2, atol=1e-8, rtol=1e-6)
    ok_l, mad_l, _, _ = compare_arrays(losses1, losses2, atol=1e-8, rtol=1e-6)
    ok_meta = all(meta1.get(k) == meta2.get(k) for k in meta1.keys()) and all(meta2.get(k) == meta1.get(k) for k in meta2.keys())

    log_result('run_proximal_on_rna_data', ok_q and ok_l and ok_meta, metric=f"Q_diff={mad_q:.3e}, loss_diff={mad_l:.3e}")
except Exception:
    log_result('run_proximal_on_rna_data', False, notes=traceback.format_exc())

try:
    df1, best1 = base.run_proximal_hyperparameter_sweep(
        folder_path=data_folder,
        rna_key_contains='rna',
        lam_values=(0.05, 0.1),
        gamma_values=(0.01,),
        eta_values=(0.5,),
        max_iter_values=(6,),
        top_n_genes_values=(30,),
        sort_by='final_loss',
        ascending=True,
        make_best_plot=False,
    )
    df2, best2 = numba_mod.run_proximal_hyperparameter_sweep(
        folder_path=data_folder,
        rna_key_contains='rna',
        lam_values=(0.05, 0.1),
        gamma_values=(0.01,),
        eta_values=(0.5,),
        max_iter_values=(6,),
        top_n_genes_values=(30,),
        sort_by='final_loss',
        ascending=True,
        make_best_plot=False,
    )

    common_cols = [c for c in df1.columns if c in df2.columns]
    df1c = df1[common_cols].reset_index(drop=True).copy()
    df2c = df2[common_cols].reset_index(drop=True).copy()

    numeric_cols = [c for c in common_cols if pd.api.types.is_numeric_dtype(df1c[c]) and pd.api.types.is_numeric_dtype(df2c[c])]
    non_numeric_cols = [c for c in common_cols if c not in numeric_cols]

    ok_num = True
    max_num_diff = 0.0
    for c in numeric_cols:
        arr1 = df1c[c].to_numpy(dtype=float)
        arr2 = df2c[c].to_numpy(dtype=float)
        if arr1.shape != arr2.shape:
            ok_num = False
            max_num_diff = np.inf
            break
        d = float(np.nanmax(np.abs(arr1 - arr2))) if arr1.size else 0.0
        max_num_diff = max(max_num_diff, d)
        if not np.allclose(arr1, arr2, atol=1e-8, rtol=1e-6, equal_nan=True):
            ok_num = False

    ok_non_num = all(df1c[c].equals(df2c[c]) for c in non_numeric_cols)

    best_ok = True
    if (best1 is None) != (best2 is None):
        best_ok = False
    elif best1 is not None and best2 is not None:
        bq_ok, bq_diff, _, _ = compare_arrays(best1['Q'], best2['Q'], atol=1e-8, rtol=1e-6)
        bl_ok, bl_diff, _, _ = compare_arrays(best1['losses'], best2['losses'], atol=1e-8, rtol=1e-6)
        best_ok = bq_ok and bl_ok

    log_result('run_proximal_hyperparameter_sweep', ok_num and ok_non_num and best_ok, metric=f"df_max_num_diff={max_num_diff:.3e}")
except Exception:
    log_result('run_proximal_hyperparameter_sweep', False, notes=traceback.format_exc())

try:
    A1, L1, R1 = base.run_spd_lasso_symmetry_check_on_rna_ai(top_n_genes=40, lam=0.08, gamma=0.01, eta=0.5, max_iter=8)
    A2, L2, R2 = numba_mod.run_spd_lasso_symmetry_check_on_rna_ai(top_n_genes=40, lam=0.08, gamma=0.01, eta=0.5, max_iter=8)
    ok_a, da, _, _ = compare_arrays(A1, A2, atol=1e-8, rtol=1e-6)
    ok_l, dl, _, _ = compare_arrays(L1, L2, atol=1e-8, rtol=1e-6)
    ok_r, _ = compare_dict_numeric(R1, R2, atol=1e-8, rtol=1e-6)
    log_result('run_spd_lasso_symmetry_check_on_rna_ai', ok_a and ok_l and ok_r, metric=f"A_diff={da:.3e}, L_diff={dl:.3e}")
except Exception:
    log_result('run_spd_lasso_symmetry_check_on_rna_ai', False, notes=traceback.format_exc())

starting proximal solver...
1
2
3
4
5
6
7
8
9
10
starting proximal solver...
Number of CSV files found: 4

CSV files found:
  - overlap_filtered_k20me3_m_v2.csv
  - overlap_filtered_k27me3_m_v2.csv
  - overlap_filtered_k9me2_m_v2.csv
  - overlap_filtered_rna_ai_m_v2.csv
  Filtered to integer time points: ['0', '1', '2', '3', '4']
Imported 'overlap_filtered_k20me3_m_v2' with shape: (657, 5) (row-wise standardized)
  Filtered to integer time points: ['0', '1', '2', '3', '4']
Imported 'overlap_filtered_k27me3_m_v2' with shape: (650, 5) (row-wise standardized)
  Filtered to integer time points: ['0', '1', '2', '3', '4']
Imported 'overlap_filtered_k9me2_m_v2' with shape: (1001, 5) (row-wise standardized)
  Filtered to integer time points: ['0', '1', '2', '3', '4']
Imported 'overlap_filtered_rna_ai_m_v2' with shape: (384, 5) (row-wise standardized)

Successfully imported 4 datasets.
Available datasets: ['overlap_filtered_k20me3_m_v2', 'overlap_filtered_k27me3_m_v2', 'overlap_filtered_k9me2_m

In [7]:
# Summary report.
results_df = pd.DataFrame(results)
if results_df.empty:
    raise RuntimeError('No results captured')

results_df = results_df[['function', 'passed', 'metric', 'notes']]
n_pass = int(results_df['passed'].sum())
n_total = int(len(results_df))

display(results_df)
print(f'Passed: {n_pass}/{n_total}')

if n_pass != n_total:
    print('Some checks failed. Inspect the notes column for traceback/details.')
else:
    print('All compared functions matched within tolerance.')

,function,passed,metric,notes
0,test_adjacency_symmetry,True,,"{'is_symmetric': True, 'max_abs_asymmetry': 0...."
1,test_adjacency_positive_semidefinite,True,,min_eig_base=-2.185e+00
2,get_adjacency_element_range,True,,
3,f,True,abs_diff=0.000e+00,
4,g,True,abs_diff=0.000e+00,
5,PI,True,"max_abs_diff=0.000e+00, shape_base=(12, 12), s...",
6,gradient_f,True,"max_abs_diff=5.684e-14, shape_base=(12, 12), s...",
7,next_omega,True,"max_abs_diff=0.000e+00, shape_base=(12, 12), s...",
8,prox_g_over_eta,True,"max_abs_diff=0.000e+00, shape_base=(12, 12), s...",
9,dual_update,True,"max_abs_diff=0.000e+00, shape_base=(12, 12), s...",


Passed: 15/15
All compared functions matched within tolerance.
